## Building an Autonomous Claude Agent

Here is the complete lesson text beautifully formatted in clean, professional Markdown (`.md`), optimized with code blocks and structural highlights for readability:

---

Throughout this course, you mastered the fundamentals of tool integration with Claude: creating tool schemas, understanding Claude's tool use responses, and executing single tool requests. However, the approach you've learned so far has a significant limitation — it only handles one tool call per conversation turn. While this works perfectly for simple tasks, many real-world problems require multiple sequential steps, and often the number and nature of these steps cannot be determined in advance.

In this lesson, we'll work together to transform Claude from a single-turn tool user into an autonomous agent capable of iterative problem-solving. We'll build an agent class that can call tools, analyze results, decide what to do next, and continue this process until complex multi-step tasks are completed. This represents a fundamental shift from reactive tool usage to proactive, intelligent problem-solving that mirrors how humans approach complex challenges.

### The Action-Feedback Loop Concept

Before we start coding, let's understand how autonomous agents operate through action-feedback loops in which each tool execution provides information that influences the next decision. This iterative process mirrors human problem-solving: we take an action, observe the result, decide what to do next, and repeat until we reach our goal. The action-feedback loop consists of four key phases that repeat until task completion:

1. **Decision Phase:** Claude analyzes the current situation and determines the next action, which may include calling one or more tools.
2. **Action Phase:** Our agent executes the requested tool(s) based on Claude's instructions.
3. **Feedback Phase:** The results from the tool execution(s) are captured and added to the conversation history.
4. **Evaluation Phase:** Claude reviews the new information, decides whether the task is complete, or if additional steps are needed, and the loop continues.

This loop structure enables complex problem-solving because each iteration builds upon previous results. For example, when solving a quadratic equation, Claude might first calculate the discriminant, then use that result to determine if real solutions exist, then calculate the square root of the discriminant, and finally compute the two solutions.

The key insight is that Claude doesn't need to plan all steps in advance — it can adapt its approach based on intermediate results, just like a human mathematician working through a problem.

Now let's start building our agent class to make this iterative process possible.

---

### Building Our Agent Class Foundation

Let's begin by creating the foundation of our autonomous agent. We need to establish the core structure that will manage extended conversations, tool execution, and decision-making loops. We'll start with the class definition and constructor:

```python
import json
import anthropic

class Agent:
    # Base system prompt to be used for all agents
    BASE_SYSTEM_PROMPT = (
        "You are an autonomous agent that can take multiple tool-calling steps when helpful. "
        "The user only sees your response when you stop using tools, not your tool usage or reasoning steps. "
        "When you provide your answer without calling tools, make it complete and standalone.\n"
        "Additional instructions:\n"
    )
    
    def __init__(
        self,
        name,
        system_prompt="You are a helpful assistant.",
        model="claude-sonnet-4-6",
        tools=None,
        tool_schemas=None,
        max_turns=10
    ):
        self.client = anthropic.Anthropic()
        self.name = name
        self.model = model
        self.system_prompt = self.BASE_SYSTEM_PROMPT + system_prompt
        self.max_turns = max_turns
        # Avoid shared mutable defaults and protect against external mutation
        self.tools = {} if tools is None else dict(tools)                # name -> Python function
        self.tool_schemas = [] if tool_schemas is None else list(tool_schemas)  # list of {name, description, input_schema}

```

Our agent's foundation relies on key design decisions that enable autonomous behavior while maintaining flexibility for different use cases:

* **`BASE_SYSTEM_PROMPT`**: Explicitly tells Claude that it can make multiple tool calls and that users won't see the intermediate steps — only the final result. We're combining this with a custom `system_prompt` to allow for domain-specific instructions while maintaining the autonomous behavior.
* **Constructor parameters**: Provide flexibility for different scenarios while ensuring some safe defaults:
* **`name`**: Provides a clear identifier for the agent, useful for debugging, logging, and when working with multiple agents in complex systems.
* **`system_prompt`**: Allows customization for specific domains like math or data analysis.
* **`model`**: Specifies which Claude model to use for agent interactions.
* **`tools` and `tool_schemas**`: Default to `None` to avoid shared mutable defaults. Copied via `dict(...)` and `list(...)` so each agent gets its own independent registry and schema list, preventing later external mutations from affecting the agent. They accept any mapping/iterable-compatible inputs for convenience.
* **`max_turns`**: Prevents infinite loops by limiting the number of iterative steps.



This architecture separates concerns cleanly while preparing us to implement the core functionality that will make our agent truly autonomous.

---

### Adding Helper Methods for State Management

As our agent works through complex problems, we need to manage conversation state properly. Let's add two essential helper methods that will support our main loop:

```python
    def _extract_text(self, content):
        # Return a joined string of all text blocks from content
        return "".join(
            block.text for block in content
            if getattr(block, "type", None) == "text"
        )

    def _build_request_args(self, messages):
        # Create a dictionary with the basic request arguments
        request_args = {
            "model": self.model,
            "system": self.system_prompt,
            "messages": messages,
            "max_tokens": 8000,
        }
        # Add tool schemas only if they exist
        if self.tool_schemas:
            request_args["tools"] = self.tool_schemas
        # Return the complete set of arguments to use for the API call
        return request_args

```

These helper methods might seem simple, but they're essential for maintaining clean separation between the complex orchestration logic we're about to write and the details of message handling:

* **`_extract_text`**: Safely extracts and combines text blocks from Claude's responses, which could contain one or more text blocks mixed with other content like tool use blocks. This ensures we return clean, readable final responses to users.
* **`_build_request_args`**: Centralizes how we construct API requests, ensuring consistent parameters across all agent interactions. Notice how we conditionally include tool schemas — this prevents API errors when we create agents without tools while still supporting full tool integration when needed.

---

### Implementing Tool Execution

Now let's add the method that handles individual tool executions within our agent loop. This method needs to be robust because tool failures shouldn't break our entire autonomous process:

```python
    def call_tool(self, tool_use):
        # Get the tool name, input, and id
        tool_name = tool_use.name
        tool_input = tool_use.input or {}
        tool_use_id = tool_use.id
        
        # Display which tool is being called
        print(f"🔧 Tool called: {tool_name}({tool_input})")
        
        try:
            # Execute the tool with the input the given input
            result = str(self.tools[tool_name](**tool_input))
        except KeyError:
            # Return an error message if the tool is not found
            result = f"Error: Tool {tool_name} not found"
        except Exception as e:
            # Return an error message if the tool fails
            result = f"Error: {str(e)}"
            
        # Return the tool result
        return {
            "type": "tool_result",
            "tool_use_id": tool_use_id,
            "content": result
        }

```

This method handles the individual tool executions that will happen within our larger iterative loop. Here's how it manages the execution flow:

* **Extract tool information:** Gets the tool name, input parameters, and unique ID from Claude's tool use request.
* **Debug tracking:** Prints which tool is being called with what parameters — invaluable for debugging and understanding how our agent thinks.
* **Execute with comprehensive error handling:** Attempts to run the tool, catching both missing tools (`KeyError` - equivalent to checking if the tool exists in our dictionary) and execution failures (`Exception`).
* **Return structured results:** Converts both successful results and errors into properly formatted tool result objects.

The error handling ensures that tool failures don't break the entire agent loop. Instead, errors are converted into tool results that Claude can understand and potentially work around. This robustness allows our agent to continue operating even when individual tools encounter problems, making the whole system much more resilient.

---

### Building the Core Loop - Part 1: Understanding Stateless Design

Now we're ready to implement the heart of our autonomous agent: the `run` method. This method will manage the iterative loop that enables multi-step problem-solving. Let's start by understanding how our agent handles conversation state:

```python
    def run(self, input_messages):
        # Create a copy of the input messages to avoid modifying the original
        messages = input_messages.copy()

```

The `input_messages.copy()` is important because it ensures our agent remains **stateless**. Just like normal LLM API calls where you pass the complete conversation history each time, our agent doesn't store any conversation state between calls. Each time you call `agent.run()`, you provide the full context through `input_messages`, and the agent processes only that specific conversation without any memory of previous interactions.

By copying the input messages instead of modifying them directly, we preserve the original conversation and allow the same agent instance to handle multiple independent conversations. This design also gives you complete control over context management — you can decide exactly what conversation history to include, filter out irrelevant messages, or combine conversations as needed before passing them to the agent.

---

### Building the Core Loop - Part 2: Setting Up the Iteration

Now let's add the basic loop structure that will enable our agent's iterative problem-solving:

```python
    def run(self, input_messages):
        # Create a copy of the input messages to avoid modifying the original
        messages = input_messages.copy()
        
        # Initialize turn counter to track iterations
        turn = 0
        
        # Loop until the model returns a final answer or the max turns is reached
        while turn < self.max_turns:
            # Increment the turn
            turn += 1
            
            # Ask the model for a response
            response = self.client.messages.create(**self._build_request_args(messages))
            
            # Add Claude's response to messages exactly as returned (text + tool_use blocks)
            messages.append({"role": "assistant", "content": response.content})

```

We're starting with a controlled loop that will continue until Claude provides a final answer or we reach our maximum turn limit. Each iteration represents one complete action-feedback cycle where Claude makes a decision (potentially including tool calls), and we capture that decision in our conversation history. The turn counter prevents infinite loops while allowing sufficient iterations for complex problems.

---

### Building the Core Loop - Part 3: Handling Tool Calls

Now let's add the logic for handling tool calls within our loop. This is where the magic of autonomous behavior happens:

```python
            # Check if Claude wants to use any tools
            if response.stop_reason == "tool_use":
                # Initialize a list to store tool results
                tool_results = []
                
                # Execute each tool use
                for content_item in response.content:
                    # Check if the content item is a tool use
                    if content_item.type == "tool_use":
                        # Execute the tool with the input the given input
                        tool_result = self.call_tool(content_item)
                        # Add result to tool results list
                        tool_results.append(tool_result)
                        
                # Add all tool results to messages
                messages.append({
                    "role": "user",
                    "content": tool_results
                })

```

When Claude decides to use tools, we handle the execution through a systematic process:

* **Execute all requested tools:** Claude might call multiple tools in a single turn, and we need to execute each one to gather all the information it needs for its next decision.
* **Collect results systematically:** We iterate through all content items in Claude's response, identify tool use blocks, and execute each tool while collecting the results in a list.
* **Feed results back as user message:** By adding the tool results as a `"user"` message, we maintain the proper conversation flow that Claude expects, ensuring the results become part of the context for the next iteration.

Each tool result influences Claude's subsequent reasoning, allowing it to build upon what it just learned and make more informed decisions in the next turn.

---

### Building the Core Loop - Part 4: Managing Flow Control

Finally, let's complete our loop with the logic for handling final responses and error conditions:

```python
            else:
                # Extract the text from the response
                response_text = self._extract_text(response.content)
                # Return the agent history and final output
                return messages, response_text
                
        # If the max turns is reached, raise an exception
        raise Exception("Max turns reached")

```

When Claude reaches a final answer, we handle the completion through a structured return process:

* **Detect completion:** When Claude doesn't want to use tools (indicated by a different `stop_reason`), it signals that it has reached a final answer and no further iterations are needed.
* **Extract clean response:** We use our `_extract_text` helper to pull out the readable text content from Claude's response, filtering out any non-text blocks.
* **Return complete state:** We return both the full conversation history (`messages`) and the final response text (`response_text`) to maintain our stateless design — the caller receives everything needed to understand what happened and can use the conversation history for follow-up questions or multi-turn interactions.
* **Safety net for runaway loops:** The exception for reaching max turns prevents infinite loops if something goes wrong. You can control this limit through the `max_turns` parameter, or alternatively implement a mechanism to force Claude to provide a final answer when approaching the limit rather than raising an exception.

This dual return approach reinforces our stateless architecture by giving the caller complete control over conversation state while providing both the detailed interaction history for continued conversations and the clean final answer for immediate use.

---

### Complete Run Method

Here's how our complete `run` method looks when put together:

```python
    def run(self, input_messages):
        # Create a copy of the input messages to avoid modifying the original
        messages = input_messages.copy()
        
        # Initialize turn counter to track iterations
        turn = 0
        
        # Loop until the model returns a final answer or the max turns is reached
        while turn < self.max_turns:
            # Increment the turn
            turn += 1
            
            # Ask the model for a response
            response = self.client.messages.create(**self._build_request_args(messages))
            
            # Add Claude's response to messages exactly as returned (text + tool_use blocks)
            messages.append({"role": "assistant", "content": response.content})
            
            # Check if Claude wants to use any tools
            if response.stop_reason == "tool_use":
                # Initialize a list to store tool results
                tool_results = []
                # Execute each tool use
                for content_item in response.content:
                    # Check if the content item is a tool use
                    if content_item.type == "tool_use":
                        # Execute the tool with the input the given input
                        tool_result = self.call_tool(content_item)
                        # Add result to tool results list
                        tool_results.append(tool_result)
                # Add all tool results to messages
                messages.append({
                    "role": "user",
                    "content": tool_results
                })
            else:
                # Extract the text from the response
                response_text = self._extract_text(response.content)
                # Return the agent history and final output
                return messages, response_text
                
        # If the max turns is reached, raise an exception
        raise Exception("Max turns reached")

```

---

### Testing Our Autonomous Agent

Now let's put our agent to work! We'll create a math-focused autonomous agent and see how it handles a complex quadratic equation. We'll provide more math tools following the same pattern used across the course, so you can easily extend your agent's capabilities as needed:

```python
import json
from agent import Agent
from functions import sum_numbers, multiply_numbers, subtract_numbers, divide_numbers, power, square_root

# Load the schemas from JSON file
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Create a dictionary mapping tool names to functions
tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}

# Create a stateless autonomous agent
agent = Agent(
    name="math_assistant",
    system_prompt="You are a helpful math assistant.",
    tools=tools,
    tool_schemas=tool_schemas,
    max_turns=15
)

# Initialize conversation with user message
messages = [{"role": "user", "content": "Solve this equation: 2x² - 7x + 3 = 0"}]

# Send message to the stateless agent
messages, result = agent.run(messages)

# Display the response
print("\nFinal response:")
print(result)

```

When we run this code, our agent demonstrates sophisticated autonomous reasoning:

```text
🔧 Tool called: power({'base': -7, 'exponent': 2})
🔧 Tool called: multiply_numbers({'a': 4, 'b': 2})
🔧 Tool called: multiply_numbers({'a': 8, 'b': 3})
🔧 Tool called: subtract_numbers({'a': 49, 'b': 24})
🔧 Tool called: square_root({'number': 25})
🔧 Tool called: multiply_numbers({'a': 2, 'b': 2})
🔧 Tool called: sum_numbers({'a': 7, 'b': 5})
🔧 Tool called: divide_numbers({'a': 12, 'b': 4})
🔧 Tool called: subtract_numbers({'a': 7, 'b': 5})
🔧 Tool called: divide_numbers({'a': 2, 'b': 4})

Final response:
The solutions to the equation 2x² - 7x + 3 = 0 are:

**x = 3** and **x = 0.5** (or x = 1/2)

You can verify these solutions by substituting back into the original equation:
- For x = 3: 2(3)² - 7(3) + 3 = 18 - 21 + 3 = 0 ✓
- For x = 0.5: 2(0.5)² - 7(0.5) + 3 = 0.5 - 3.5 + 3 = 0 ✓

```

Our agent systematically applied the quadratic formula by calculating $b^2$ ($(-7)^2$), computing $4ac$ ($4 \times 2 \times 3$), finding the discriminant ($49-24$), taking the square root ($\sqrt{25}$), and finally calculating both solutions through the complete quadratic formula.

Each tool call built upon previous results, demonstrating true autonomous reasoning. The agent made 10 tool calls across multiple conversation turns, yet the user only sees the final, complete answer with verification.

---

### Summary & Practice Preparation

Together, we've successfully built an autonomous agent capable of complex, multi-step problem-solving. Our agent class encapsulates conversation management, tool execution, and iterative decision-making in a reusable structure that can tackle problems requiring dozens of sequential operations.

The architecture we created enables Claude to operate as a true autonomous agent: it can assess situations, make decisions, execute tools, learn from results, and continue iterating until complex tasks are completed. This represents a fundamental advancement from simple tool usage to intelligent, adaptive problem-solving.

In the upcoming practice exercises, you'll implement your own autonomous agents, experiment with different system prompts and tool combinations, and tackle increasingly complex multi-step problems. You'll gain hands-on experience with the debugging and optimization techniques needed for production agent systems, building upon the solid foundation we've created together.

## Setting the Agent Foundation

Now that you understand how autonomous agents work through action-feedback loops, it's time to build the foundation that makes this magic possible! In this exercise, you'll implement the core constructor of the Agent class, setting up all the essential components that enable Claude to operate as a truly autonomous problem solver.

Your main tasks for agent.py are to:

    Craft a robust BASE_SYSTEM_PROMPT that instructs Claude to behave as an autonomous agent
    Complete the __init__ method by properly initializing all instance variables
    Set up the Anthropic client, model configuration, agent name, and maximum turns limit
    Combine the your base system prompt with custom system prompt instructions
    Organize the tools and schemas avoiding shared mutable defaults:
        Set default parameters to None
        For tools: use {} if tools is None else dict(tools) to create a safe copy
        For tool_schemas: use [] if tool_schemas is None else list(tool_schemas) to create a safe copy

Once you've built the agent foundation, you'll verify your implementation works correctly by completing main.py to:

    Create a math assistant agent with a descriptive name using the provided tools, tool_schemas, and an appropriate system prompt
    Print the agent's name, model, max_turns, system_prompt, number of tools, and number of tool_schemas
    Print the full system prompt to verify that your agent constructor properly combines the base prompt with custom instructions

Complete this foundation correctly, and you'll have built the core that powers all autonomous agent behavior!

```
# agent.py
import json
import anthropic


class Agent:
    # TODO: Write a robust base system prompt that instructs Claude to act as an autonomous agent

    # TODO: Define the __init__ method that accepts the following parameters:
    # - name (required parameter for identifying the agent)
    # - system_prompt (default: "You are a helpful assistant.")
    # - model (default: "claude-sonnet-4-6")
    # - tools (default: None to avoid shared mutable defaults)
    # - tool_schemas (default: None to avoid shared mutable defaults)
    # - max_turns (default: 10)
    
        # TODO: Inside the __init__ method:
        # - Initialize the Anthropic client
        # - Set the agent name
        # - Set the model
        # - Combine BASE_SYSTEM_PROMPT with system_prompt
        # - Set max_turns
        # - Set up the tools dictionary (use {} if tools is None, otherwise dict(tools))
        # - Set up the tool_schemas list (use [] if tool_schemas is None, otherwise list(tool_schemas))

# main.py
import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load the schemas from JSON file
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Create a dictionary mapping tool names to functions
tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}

# TODO: Create a math assistant agent by passing appropriate parameters for:
# - A descriptive name for the agent (e.g., "math_assistant")
# - A system prompt that identifies the agent as a math assistant
# - The tools dictionary defined above
# - The tool_schemas loaded from the JSON file

# TODO: Print the agent's name
# TODO: Print the agent's model
# TODO: Print the agent's max_turns
# TODO: Print the agent's system_prompt
# TODO: Print the agent's number of tools
# TODO: Print the agent's number of tool schemas
```

Building a clean foundation for your `Agent` class requires defensive copy patterns (`dict` and `list` wrappers) to protect the internal registries from shared reference mutations across independent agent workflows.

Here is the complete implementation for both files, cleanly filling out all requirements:

### `agent.py`

```python
import json
import anthropic


class Agent:
    # Robust base system prompt instructing Claude on multi-step execution behavior
    BASE_SYSTEM_PROMPT = (
        "You are an autonomous agent capable of orchestrating multiple tool-calling actions "
        "iteratively to solve complex user tasks. The end-user will only see your final "
        "standalone response after you decide to stop using tools. Do not output your intermediate "
        "reasoning or tool execution steps to the user. When you provide your final answer, "
        "make it complete, accurate, and completely standalone.\n"
        "Additional instructions:\n"
    )

    def __init__(
        self,
        name,
        system_prompt="You are a helpful assistant.",
        model="claude-sonnet-4-6",
        tools=None,
        tool_schemas=None,
        max_turns=10
    ):
        # Initialize the Anthropic client
        self.client = anthropic.Anthropic()
        
        # Set core operational attributes
        self.name = name
        self.model = model
        self.max_turns = max_turns
        
        # Combine the structural base agent prompt with specific persona domain constraints
        self.system_prompt = self.BASE_SYSTEM_PROMPT + system_prompt
        
        # Defensively copy parameters to isolate runtime registries from mutation
        self.tools = {} if tools is None else dict(tools)
        self.tool_schemas = [] if tool_schemas is None else list(tool_schemas)

```

### `main.py`

```python
import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load the schemas from JSON file
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Create a dictionary mapping tool names to functions
tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}

# Create a math assistant agent instance using the new constructor architecture
agent = Agent(
    name="math_assistant",
    system_prompt="You are an expert mathematics tutor. Break down calculations cleanly and verify each math block before finalizing answers.",
    model="claude-sonnet-4-6",
    tools=tools,
    tool_schemas=tool_schemas,
    max_turns=15
)

# Verify the constructor layout by logging its operational boundaries
print(f"Agent Name: {agent.name}")
print(f"Model Engine: {agent.model}")
print(f"Max Turns Limit: {agent.max_turns}")
print(f"Registered Tool Count: {len(agent.tools)}")
print(f"Registered Schema Count: {len(agent.tool_schemas)}")
print("\n--- Consolidated System Prompt ---")
print(agent.system_prompt)

```

## Building the Tool Execution Engine

Building on your solid agent foundation, it's time to create the engine that actually executes tools and handles the messy realities of real-world function calls!

Your mission in agent.py is to build a bulletproof tool execution system by implementing the call_tool method:

    Extract the tool name, input parameters, and unique ID from Claude's tool_use requests
    Add helpful debug output so you can see exactly which tools are being called
    Implement comprehensive error handling for missing tools and execution failures
    Return properly formatted tool results that Claude can understand and work with

Once your tool execution engine is implemented, you'll test it in main.py where a math assistant agent and MockToolUse class are already set up for you. You'll complete the testing code by:

    Calling your call_tool method with the provided mock tool_use object for successful execution
    Printing the successful result to verify it works correctly
    Creating a second mock tool_use object for a non-existent tool to test error handling
    Calling your call_tool method with the error scenario and printing the result

This hands-on testing approach will give you confidence that your tool execution system is rock-solid before we integrate it into the full autonomous loop in future lessons!

```
# agent.py
import json
import anthropic


class Agent:
    # Base system prompt to be used for all agents
    BASE_SYSTEM_PROMPT = (
        "You are an autonomous agent that can take multiple tool-calling steps when helpful. "
        "The user only sees your response when you stop using tools, not your tool usage or reasoning steps. "
        "When you provide your answer without calling tools, make it complete and standalone.\n"
        "Additional instructions:\n"
    )

    def __init__(
        self,
        name,
        system_prompt="You are a helpful assistant.",
        model="claude-sonnet-4-6",
        tools=None,
        tool_schemas=None,
        max_turns=10
    ):
        self.client = anthropic.Anthropic()
        self.name = name
        self.model = model
        self.system_prompt = self.BASE_SYSTEM_PROMPT + system_prompt
        self.max_turns = max_turns
        self.tools = {} if tools is None else dict(tools)
        self.tool_schemas = [] if tool_schemas is None else list(tool_schemas)

    # TODO: Implement the call_tool method to execute tools and handle errors
        # TODO: Get the tool name from tool_use.name
        # TODO: Get the tool input from tool_use.input (use empty dict {} if None)
        # TODO: Get the tool use id from tool_use.id

        # TODO: Print which tool is being called with its input parameters
        
        # TODO: Create a try-except block to handle tool execution
            # TODO: Execute the tool by calling self.tools[tool_name] with **tool_input
            # TODO: Convert the result to string
        # TODO: Add except KeyError to handle missing tools
            # TODO: Set result to an error message about tool not found
        # TODO: Add except Exception to handle other errors
            # TODO: Set result to an error message with the exception details

        # TODO: Return a dictionary with type, tool_use_id, and content

# main.py
import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load the schemas from JSON file
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Create a dictionary mapping tool names to functions
tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}

# Create a math assistant agent
agent = Agent(
    name="math_assistant",
    system_prompt="You are a helpful math assistant.",
    tools=tools,
    tool_schemas=tool_schemas
)

# Mock tool_use class to test call_tool
class MockToolUse:
    def __init__(self, name, input_data, id):
        self.name = name
        self.input = input_data
        self.id = id

# Mock tool_use object to test call_tool
mock_tool_use = MockToolUse(
    name="sum_numbers",
    input_data={"a": 5, "b": 3},
    id="test_123"
)

# TODO: Call agent.call_tool with the mock tool_use object and store the result

# TODO: Print the result of the successful tool call

# TODO: Create another mock tool_use object for a non-existent tool to test error handling

# TODO: Call agent.call_tool with the error mock object and store the result

# TODO: Print the error scenario result

```

Implementing robust error handling within the tool execution layer is vital for autonomous agents. If a tool fails or if Claude requests a tool that isn't loaded into the engine, the agent must catch that exception and feed the error back to Claude as a `tool_result`. This allows Claude to see what went wrong, adapt its strategy, or attempt a different route rather than crashing the runtime environment.

Here is the complete implementation for both files, filling out all requirements:

### `agent.py`

```python
import json
import anthropic


class Agent:
    # Base system prompt to be used for all agents
    BASE_SYSTEM_PROMPT = (
        "You are an autonomous agent that can take multiple tool-calling steps when helpful. "
        "The user only sees your response when you stop using tools, not your tool usage or reasoning steps. "
        "When you provide your answer without calling tools, make it complete and standalone.\n"
        "Additional instructions:\n"
    )

    def __init__(
        self,
        name,
        system_prompt="You are a helpful assistant.",
        model="claude-sonnet-4-6",
        tools=None,
        tool_schemas=None,
        max_turns=10
    ):
        self.client = anthropic.Anthropic()
        self.name = name
        self.model = model
        self.system_prompt = self.BASE_SYSTEM_PROMPT + system_prompt
        self.max_turns = max_turns
        self.tools = {} if tools is None else dict(tools)
        self.tool_schemas = [] if tool_schemas is None else list(tool_schemas)

    def call_tool(self, tool_use):
        # Extract metadata and parameters from Claude's request block
        tool_name = tool_use.name
        tool_input = tool_use.input if tool_use.input is not None else {}
        tool_use_id = tool_use.id

        # Helpful debug trace for monitoring the agent loop
        print(f"🔧 Tool called: {tool_name}({tool_input})")
        
        try:
            # Unpack the parameters dynamically into the registered Python function
            function_result = self.tools[tool_name](**tool_input)
            result = str(function_result)
        except KeyError:
            # Explicit handling for when Claude tries to invoke a tool that doesn't exist
            result = f"Error: Tool '{tool_name}' not found in the agent's available toolset."
        except Exception as e:
            # Fallback catch for unexpected execution/runtime exceptions inside the tool
            result = f"Error executing tool '{tool_name}': {str(e)}"

        # Return a structured dictionary conforming to the API's tool_result block schema
        return {
            "type": "tool_result",
            "tool_use_id": tool_use_id,
            "content": result
        }

```

### `main.py`

```python
import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load the schemas from JSON file
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Create a dictionary mapping tool names to functions
tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}

# Create a math assistant agent
agent = Agent(
    name="math_assistant",
    system_prompt="You are a helpful math assistant.",
    tools=tools,
    tool_schemas=tool_schemas
)

# Mock tool_use class to test call_tool
class MockToolUse:
    def __init__(self, name, input_data, id):
        self.name = name
        self.input = input_data
        self.id = id

# Mock tool_use object to test call_tool (Valid case)
mock_tool_use = MockToolUse(
    name="sum_numbers",
    input_data={"a": 5, "b": 3},
    id="test_123"
)

# Test Scenario 1: Execute a valid tool call
print("=== Scenario 1: Executing a valid tool call ===")
success_result = agent.call_tool(mock_tool_use)
print("\nReturned tool_result block structure:")
print(json.dumps(success_result, indent=4))

# Mock tool_use object for a non-existent tool to test error handling
mock_error_tool_use = MockToolUse(
    name="integrate_calculus",
    input_data={"expression": "3x^2 + 2x", "variable": "x"},
    id="test_456"
)

# Test Scenario 2: Execute an invalid tool call to verify catch behavior
print("\n=== Scenario 2: Handling a non-existent tool call ===")
error_result = agent.call_tool(mock_error_tool_use)
print("\nReturned tool_result block structure:")
print(json.dumps(error_result, indent=4))

```

## Understanding Basic Stateless Design

Now that you have a solid agent foundation and robust tool execution capabilities, it's time to implement the core conversation mechanics that power every agent interaction.

In this exercise, you'll build a simplified version of the run method in agent.py that handles basic conversation flow, focusing on the fundamental request-response cycle that forms the backbone of all agent interactions:

    Preserve input state: Copy the input messages to maintain stateless behavior - never modify the original message list
    Prepare API request: Use the _build_request_args helper method to create properly formatted request arguments
    Call Claude: Make a single API call to get Claude's response using the Anthropic client
    Update conversation: Add Claude's response to the message history with the correct role and structure
    Extract response text: Use the _extract_text helper to get clean, readable text from Claude's response
    Return results: Provide both the updated conversation history and the extracted response text

Once your basic conversation flow works, test it in main.py by creating an agent for general conversation, sending two sequential messages to demonstrate how conversation history builds up, and verifying that the stateless design works correctly (original messages remain unchanged).

This foundational implementation will prepare you for more advanced agent behaviors like multi-turn conversations and autonomous tool usage in future exercises.

```
# agent.py
import json
import anthropic


class Agent:
    # Base system prompt to be used for all agents
    BASE_SYSTEM_PROMPT = (
        "You are an autonomous agent that can take multiple tool-calling steps when helpful. "
        "The user only sees your response when you stop using tools, not your tool usage or reasoning steps. "
        "When you provide your answer without calling tools, make it complete and standalone.\n"
        "Additional instructions:\n"
    )

    def __init__(
        self,
        name,
        system_prompt="You are a helpful assistant.",
        model="claude-sonnet-4-6",
        tools=None,
        tool_schemas=None,
        max_turns=10
    ):
        self.client = anthropic.Anthropic()
        self.name = name
        self.model = model
        self.system_prompt = self.BASE_SYSTEM_PROMPT + system_prompt
        self.max_turns = max_turns
        self.tools = {} if tools is None else dict(tools)
        self.tool_schemas = [] if tool_schemas is None else list(tool_schemas)
        
    def _extract_text(self, content):
        # Return a joined string of all text blocks from content
        return "".join(
            block.text for block in content
            if getattr(block, "type", None) == "text"
        )    

    def _build_request_args(self, messages):
        # Create a dictionary with the basic request arguments
        request_args = {
            "model": self.model,
            "system": self.system_prompt,
            "messages": messages,
            "max_tokens": 8000,
        }

        # Add tool schemas only if they exist
        if self.tool_schemas:
            request_args["tools"] = self.tool_schemas

        # Return the complete set of arguments to use for the API call
        return request_args

    def call_tool(self, tool_use):
        # Get the tool name, input, and id
        tool_name = tool_use.name
        tool_input = tool_use.input or {}
        tool_use_id = tool_use.id

        # Display which tool is being called
        print(f"🔧 Tool called: {tool_name}({tool_input})")
        
        try:
            # Execute the tool with the input the given input
            result = str(self.tools[tool_name](**tool_input))
        except KeyError:
            # Return an error message if the tool is not found
            result = f"Error: Tool {tool_name} not found"
        except Exception as e:
            # Return an error message if the tool fails
            result = f"Error: {str(e)}"

        # Return the tool result
        return {
            "type": "tool_result",
            "tool_use_id": tool_use_id,
            "content": result
        }

    # TODO: Implement the run method that takes a list of conversation messages
    
        # TODO: Create a copy of the input messages to avoid modifying the original

        # TODO: Make an API call to Claude using self.client.messages.create() with the _build_request_args helper

        # TODO: Append Claude's response to messages with role "assistant" and content from response.content

        # TODO: Extract the response text using the _extract_text helper method

        # TODO: Return both the updated messages and the extracted response text

# main.py
from agent import Agent

# TODO: Create a simple agent for general conversation with an appropriate system prompt

# TODO: Start with a message list containing one user message

# TODO: Send the first message to the agent and print the response

# TODO: Add a follow-up user question to the conversation history

# TODO: Send the updated conversation to the agent and print the response

```

Implementing a **stateless design** ensures that the agent context is controlled strictly by the parameters passed into it. By using `input_messages.copy()`, the agent instance remains re-usable and free from side effects, mirroring standard stateless REST API patterns.

Here is the complete implementation for both files, filling out all requirements:

### `agent.py`

```python
import json
import anthropic


class Agent:
    # Base system prompt to be used for all agents
    BASE_SYSTEM_PROMPT = (
        "You are an autonomous agent that can take multiple tool-calling steps when helpful. "
        "The user only sees your response when you stop using tools, not your tool usage or reasoning steps. "
        "When you provide your answer without calling tools, make it complete and standalone.\n"
        "Additional instructions:\n"
    )

    def __init__(
        self,
        name,
        system_prompt="You are a helpful assistant.",
        model="claude-sonnet-4-6",
        tools=None,
        tool_schemas=None,
        max_turns=10
    ):
        self.client = anthropic.Anthropic()
        self.name = name
        self.model = model
        self.system_prompt = self.BASE_SYSTEM_PROMPT + system_prompt
        self.max_turns = max_turns
        self.tools = {} if tools is None else dict(tools)
        self.tool_schemas = [] if tool_schemas is None else list(tool_schemas)
        
    def _extract_text(self, content):
        # Return a joined string of all text blocks from content
        return "".join(
            block.text for block in content
            if getattr(block, "type", None) == "text"
        )    

    def _build_request_args(self, messages):
        # Create a dictionary with the basic request arguments
        request_args = {
            "model": self.model,
            "system": self.system_prompt,
            "messages": messages,
            "max_tokens": 8000,
        }

        # Add tool schemas only if they exist
        if self.tool_schemas:
            request_args["tools"] = self.tool_schemas

        # Return the complete set of arguments to use for the API call
        return request_args

    def call_tool(self, tool_use):
        # Get the tool name, input, and id
        tool_name = tool_use.name
        tool_input = tool_use.input or {}
        tool_use_id = tool_use.id

        # Display which tool is being called
        print(f"🔧 Tool called: {tool_name}({tool_input})")
        
        try:
            # Execute the tool with the input the given input
            result = str(self.tools[tool_name](**tool_input))
        except KeyError:
            # Return an error message if the tool is not found
            result = f"Error: Tool {tool_name} not found"
        except Exception as e:
            # Return an error message if the tool fails
            result = f"Error: {str(e)}"

        # Return the tool result
        return {
            "type": "tool_result",
            "tool_use_id": tool_use_id,
            "content": result
        }

    def run(self, input_messages):
        # Create a copy of the input messages to guarantee stateless behavior
        messages = input_messages.copy()

        # Build request parameters and send context payload to Claude
        request_args = self._build_request_args(messages)
        response = self.client.messages.create(**request_args)

        # Append Claude's raw response block array to the message log
        messages.append({
            "role": "assistant",
            "content": response.content
        })

        # Extract textual information to generate clean conversational responses
        response_text = self._extract_text(response.content)

        # Return the augmented conversation history log along with clean textual answers
        return messages, response_text

```

### `main.py`

```python
from agent import Agent

# Create a simple agent for general conversation
agent = Agent(
    name="chat_bot",
    system_prompt="You are a concise assistant. Provide answers in 1 or 2 clear sentences max."
)

# Start with a message list containing one user message
initial_history = [
    {"role": "user", "content": "Hi! What is the capital of France?"}
]

# Send the first message to the agent and process the output tuple
print("--- Round 1 ---")
updated_history_1, response_1 = agent.run(initial_history)
print(f"Claude: {response_1}")

# Verify that the initial list object remains untouched (Stateless check)
assert len(initial_history) == 1, "Error: The initial history was accidentally modified!"

# Add a follow-up user question to the accumulated runtime log history
updated_history_1.append(
    {"role": "user", "content": "Great. What is its most famous tower?"}
)

# Send the updated multi-turn conversation stack to the agent
print("\n--- Round 2 ---")
final_history, response_2 = agent.run(updated_history_1)
print(f"Claude: {response_2}")

# Verify structural separation across variables
assert len(updated_history_1) == 3, "Error: Second round side effects affected Turn 1 data!"
print(f"\nFinal tracking logs count: {len(final_history)} items across conversation loop.")

```


## Implementing the Autonomous Agent Loop

## Completing the Autonomous Agent Loop

## Testing Agent Limits with Complex Equations